# 01 — Setup and data validation

Run this notebook first with `dtp_fact_kaggle_dataset_2026-10-25.zip`
attached as a private Kaggle Dataset. It installs the background package,
runs every software test, executes the synthetic integration diagnostic,
and checks the collected candidate snapshot.

**Boundary:** the included candidates are screening inputs, not audited gold
data. Passing this notebook validates software and collection integrity only;
it does not validate the paper's scientific findings.


In [1]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

ON_KAGGLE = Path("/kaggle/input").exists()
INPUT_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path.cwd()
WORK_ROOT = Path("/kaggle/working/dtp_fact_runtime") if ON_KAGGLE else Path.cwd() / "_dtp_fact_runtime"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def find_background():
    candidates = sorted(INPUT_ROOT.rglob("background/pyproject.toml"))
    candidates += sorted(WORK_ROOT.rglob("background/pyproject.toml"))
    if not candidates:
        archives = sorted(INPUT_ROOT.rglob("dtp_fact_kaggle_dataset*.zip"))
        if not archives:
            raise FileNotFoundError(
                "Attach dtp_fact_kaggle_dataset_2026-10-25.zip as a Kaggle Dataset."
            )
        extract_root = WORK_ROOT / "dataset"
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall(extract_root)
        candidates = sorted(extract_root.rglob("background/pyproject.toml"))
    if not candidates:
        raise FileNotFoundError("The attached dataset has no background/pyproject.toml")
    return candidates[0].parent, candidates[0].parent.parent

CODE_ROOT, BUNDLE_ROOT = find_background()
DATA_ROOT = BUNDLE_ROOT / "data"
sys.path.insert(0, str(CODE_ROOT / "src"))

def install_background():
    install_command = [
        sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
        "-e", f"{CODE_ROOT}[analysis,test,retrieval,ml]",
    ]
    subprocess.run(install_command, check=True)
    print("Background package installed from", CODE_ROOT)

print(json.dumps({
    "on_kaggle": ON_KAGGLE,
    "code_root": str(CODE_ROOT),
    "data_root": str(DATA_ROOT),
    "work_root": str(WORK_ROOT),
}, indent=2))


{
  "on_kaggle": false,
  "code_root": "D:\\1.msc\\ET2\\kaggle\\build-2026-10-25\\notebook-run-v1\\_dtp_fact_runtime\\dataset\\dtp_fact_kaggle_dataset\\background",
  "data_root": "D:\\1.msc\\ET2\\kaggle\\build-2026-10-25\\notebook-run-v1\\_dtp_fact_runtime\\dataset\\dtp_fact_kaggle_dataset\\data",
  "work_root": "D:\\1.msc\\ET2\\kaggle\\build-2026-10-25\\notebook-run-v1\\_dtp_fact_runtime"
}


In [2]:
install_background()


Background package installed from D:\1.msc\ET2\kaggle\build-2026-10-25\notebook-run-v1\_dtp_fact_runtime\dataset\dtp_fact_kaggle_dataset\background


In [3]:
test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=CODE_ROOT,
    text=True,
    capture_output=True,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode != 0:
    raise RuntimeError("The background package tests failed.")

smoke_run = subprocess.run(
    [sys.executable, "scripts/run_smoke_test.py", "--config", "configs/base.yaml"],
    cwd=CODE_ROOT,
    text=True,
    capture_output=True,
)
print(smoke_run.stdout)
if smoke_run.stderr:
    print(smoke_run.stderr)
if smoke_run.returncode != 0:
    raise RuntimeError("The synthetic integration diagnostic failed.")

smoke_metrics = json.loads((CODE_ROOT / "outputs/smoke/metrics.json").read_text(encoding="utf-8"))
assert smoke_metrics["status"] == "synthetic_software_diagnostic_only"
print("Software checks passed; smoke values remain non-scientific diagnostics.")


..........................................                               [100%]
42 passed in 2.77s

{
  "accuracy": 1.0,
  "aurc": 0.0,
  "balanced_accuracy": 1.0,
  "brier": 0.05999999999999999,
  "ece": 0.20000000000000007,
  "macro_f1": 1.0,
  "macro_ovr_auroc": 1.0,
  "nll": 0.2231435513142097,
  "per_class": {
    "insufficient_evidence": {
      "f1": 1.0,
      "precision": 1.0,
      "recall": 1.0
    },
    "refuted": {
      "f1": 1.0,
      "precision": 1.0,
      "recall": 1.0
    },
    "supported": {
      "f1": 1.0,
      "precision": 1.0,
      "recall": 1.0
    }
  },
  "risk_at_80_coverage": 0.0,
  "sample_count": 6,
  "split_counts": {
    "test": 3,
    "train": 2,
    "validation": 1
  },
  "status": "synthetic_software_diagnostic_only"
}



Software checks passed; smoke values remain non-scientific diagnostics.


In [4]:
import csv
from collections import Counter

snapshots = sorted(DATA_ROOT.rglob("collection_manifest.json"))
if not snapshots:
    raise FileNotFoundError("Candidate collection_manifest.json is absent.")
candidate_root = snapshots[0].parent
collection = json.loads((candidate_root / "collection_manifest.json").read_text(encoding="utf-8"))
queue_manifest = json.loads((candidate_root / "screening_queue_manifest.json").read_text(encoding="utf-8"))

with (candidate_root / "candidates.jsonl").open("r", encoding="utf-8") as stream:
    candidate_count = sum(bool(line.strip()) for line in stream)
with (candidate_root / "screening_queue.csv").open("r", encoding="utf-8-sig", newline="") as stream:
    queue_rows = list(csv.DictReader(stream))
queue_labels = Counter(row["suggested_label"] for row in queue_rows)
raw_pages = len(list((candidate_root / "raw").glob("*.json")))

checks = {
    "candidate_count_32898": candidate_count == 32898,
    "manifest_count_matches": candidate_count == collection["candidate_count_after_deduplication"],
    "raw_pages_440": raw_pages == 440,
    "queue_size_1200": len(queue_rows) == 1200 == queue_manifest["actual_size"],
    "queue_balanced": queue_labels == Counter({
        "supported": 400, "refuted": 400, "insufficient_evidence": 400
    }),
    "all_rows_pending": all(row["decision"] == "pending" for row in queue_rows),
    "api_key_not_recorded": collection.get("api_key_recorded") is False,
    "no_secret_directory": not any(path.name == ".secrets" for path in BUNDLE_ROOT.rglob(".secrets")),
}
report = {
    "status": "candidate_snapshot_validated_for_screening_only" if all(checks.values()) else "candidate_snapshot_invalid",
    "scientific_results_validated": False,
    "candidate_count": candidate_count,
    "raw_response_pages": raw_pages,
    "screening_queue_rows": len(queue_rows),
    "suggested_label_counts": dict(queue_labels),
    "checks": checks,
    "interpretation": "Automatic suggested labels are not gold labels; do not train the final study on this snapshot.",
}
report_path = WORK_ROOT / "01_data_validation_report.json"
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2, sort_keys=True))
if not all(checks.values()):
    raise RuntimeError("Candidate snapshot validation failed.")


{
  "candidate_count": 32898,
  "checks": {
    "all_rows_pending": true,
    "api_key_not_recorded": true,
    "candidate_count_32898": true,
    "manifest_count_matches": true,
    "no_secret_directory": true,
    "queue_balanced": true,
    "queue_size_1200": true,
    "raw_pages_440": true
  },
  "interpretation": "Automatic suggested labels are not gold labels; do not train the final study on this snapshot.",
  "raw_response_pages": 440,
  "scientific_results_validated": false,
  "screening_queue_rows": 1200,
  "status": "candidate_snapshot_validated_for_screening_only",
  "suggested_label_counts": {
    "insufficient_evidence": 400,
    "refuted": 400,
    "supported": 400
  }
}


If the final line says `candidate_snapshot_validated_for_screening_only`,
the code and acquisition snapshot are intact. Continue to notebook 02 only
after an audited final-data dataset is attached. Notebook 02 will block
rather than silently treat suggested labels as scientific ground truth.
